In [1]:
from pathlib import Path
import json
import pandas as pd
import torch
from torch.utils.data import WeightedRandomSampler, DataLoader
from torchvision import datasets, transforms

PROJECT_ROOT = Path("/backup/Intern/combine_skindiseaseclassifier_devraj")
DATA_DIR = PROJECT_ROOT / "data" / "splits"
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"
TEST_DIR = DATA_DIR / "test"

REPORT_DIR = PROJECT_ROOT / "reports" / "balancing"
MD_DIR = PROJECT_ROOT / "markdown_reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MD_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
BATCH_SIZE = 32

In [2]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.80, 1.00),
        ratio=(0.90, 1.10)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.05,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

In [3]:
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=eval_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)

class_to_idx = train_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

print("Classes:", len(class_to_idx))
print("Train images:", len(train_dataset))
print("Val images:", len(val_dataset))
print("Test images:", len(test_dataset))

class_to_idx

Classes: 14
Train images: 25576
Val images: 5478
Test images: 5474


{'acne_vulgaris': 0,
 'atopic_dermatitis': 1,
 'basal_cell_carcinoma': 2,
 'contact_dermatitis': 3,
 'drug_eruptions': 4,
 'folliculitis': 5,
 'fungal_nail_infections': 6,
 'lupus_related_skin_lesions': 7,
 'melanoma': 8,
 'plaque_psoriasis': 9,
 'seborrheic_dermatitis': 10,
 'tinea_corporis': 11,
 'vitiligo': 12,
 'warts': 13}

In [5]:
targets = torch.tensor(train_dataset.targets)

class_counts = torch.bincount(targets)
num_classes = len(class_counts)

rows = []

for class_idx, count in enumerate(class_counts.tolist()):
    rows.append({
        "class_idx": class_idx,
        "class_name": idx_to_class[class_idx],
        "train_count": count
    })

class_counts_df = pd.DataFrame(rows)
class_counts_df

,class_idx,class_name,train_count
0,0,acne_vulgaris,1727
1,1,atopic_dermatitis,1376
2,2,basal_cell_carcinoma,4247
3,3,contact_dermatitis,1062
4,4,drug_eruptions,1115
5,5,folliculitis,825
6,6,fungal_nail_infections,844
7,7,lupus_related_skin_lesions,900
8,8,melanoma,6367
9,9,plaque_psoriasis,2342


In [6]:
class_counts_path = REPORT_DIR / "train_class_counts.csv"
class_counts_df.to_csv(class_counts_path, index=False)
class_counts_path

PosixPath('/backup/Intern/combine_skindiseaseclassifier_devraj/reports/balancing/train_class_counts.csv')

In [7]:
total_train_images = len(train_dataset)

class_weights = total_train_images / (num_classes * class_counts.float())

weights_rows = []

for class_idx, weight in enumerate(class_weights.tolist()):
    weights_rows.append({
        "class_idx": class_idx,
        "class_name": idx_to_class[class_idx],
        "train_count": int(class_counts[class_idx]),
        "class_weight": weight
    })

class_weights_df = pd.DataFrame(weights_rows)
class_weights_df

,class_idx,class_name,train_count,class_weight
0,0,acne_vulgaris,1727,1.057821
1,1,atopic_dermatitis,1376,1.327658
2,2,basal_cell_carcinoma,4247,0.430152
3,3,contact_dermatitis,1062,1.720205
4,4,drug_eruptions,1115,1.638437
5,5,folliculitis,825,2.214372
6,6,fungal_nail_infections,844,2.164523
7,7,lupus_related_skin_lesions,900,2.029841
8,8,melanoma,6367,0.286926
9,9,plaque_psoriasis,2342,0.780041


In [8]:
class_weights_path = REPORT_DIR / "class_weights.csv"
class_weights_df.to_csv(class_weights_path, index=False)
class_weights_path

PosixPath('/backup/Intern/combine_skindiseaseclassifier_devraj/reports/balancing/class_weights.csv')

In [9]:
sample_weights = class_weights[targets]

sample_rows = []

for image_path, label_idx, sample_weight in zip(
    train_dataset.samples,
    train_dataset.targets,
    sample_weights.tolist()
):
    sample_rows.append({
        "image_path": image_path[0],
        "class_idx": label_idx,
        "class_name": idx_to_class[label_idx],
        "sample_weight": sample_weight
    })

sample_weights_df = pd.DataFrame(sample_rows)
sample_weights_df.head()

,image_path,class_idx,class_name,sample_weight
0,/backup/Intern/combine_skindiseaseclassifier_d...,0,acne_vulgaris,1.057821
1,/backup/Intern/combine_skindiseaseclassifier_d...,0,acne_vulgaris,1.057821
2,/backup/Intern/combine_skindiseaseclassifier_d...,0,acne_vulgaris,1.057821
3,/backup/Intern/combine_skindiseaseclassifier_d...,0,acne_vulgaris,1.057821
4,/backup/Intern/combine_skindiseaseclassifier_d...,0,acne_vulgaris,1.057821


In [10]:
sample_weights_path = REPORT_DIR / "train_sample_weights.csv"
sample_weights_df.to_csv(sample_weights_path, index=False)
sample_weights_path

PosixPath('/backup/Intern/combine_skindiseaseclassifier_devraj/reports/balancing/train_sample_weights.csv')

In [12]:
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=weighted_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("Batch labels:", labels.tolist())

Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32])
Batch labels: [13, 13, 2, 4, 7, 2, 1, 1, 0, 7, 12, 6, 9, 1, 3, 10, 13, 5, 12, 8, 2, 1, 5, 6, 6, 4, 5, 6, 13, 3, 7, 1]


In [13]:
batch_class_names = [idx_to_class[int(label)] for label in labels]
pd.Series(batch_class_names).value_counts()

atopic_dermatitis             5
warts                         4
fungal_nail_infections        4
lupus_related_skin_lesions    3
basal_cell_carcinoma          3
folliculitis                  3
drug_eruptions                2
contact_dermatitis            2
vitiligo                      2
plaque_psoriasis              1
acne_vulgaris                 1
seborrheic_dermatitis         1
melanoma                      1
Name: count, dtype: int64

In [14]:
balance_config = {
    "method": "WeightedRandomSampler",
    "used_for": "train split only",
    "validation_balanced": False,
    "test_balanced": False,
    "augmentation": {
        "train": [
            "Resize(256)",
            "RandomResizedCrop(224, scale=(0.80, 1.00), ratio=(0.90, 1.10))",
            "RandomHorizontalFlip(p=0.5)",
            "RandomRotation(10)",
            "Mild ColorJitter",
            "Normalize(ImageNet)"
        ],
        "val_test": [
            "Resize(256)",
            "CenterCrop(224)",
            "Normalize(ImageNet)"
        ]
    },
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "num_classes": num_classes,
    "total_train_images": total_train_images,
}

config_path = REPORT_DIR / "balance_config.json"

with config_path.open("w", encoding="utf-8") as f:
    json.dump(balance_config, f, indent=2)

config_path

PosixPath('/backup/Intern/combine_skindiseaseclassifier_devraj/reports/balancing/balance_config.json')

In [15]:
md_path = MD_DIR / "balancing_and_augmentation_summary.md"

with md_path.open("w", encoding="utf-8") as f:
    f.write("# Balancing and Augmentation Summary\n\n")

    f.write("## Balancing Method\n\n")
    f.write("For PyTorch models, balancing is done using `WeightedRandomSampler` on the training split only.\n\n")
    f.write("No images are duplicated or physically copied for PyTorch training.\n\n")

    f.write("## Why Only Train Split Is Balanced\n\n")
    f.write("Validation and test splits are kept unchanged so evaluation remains realistic and honest.\n\n")

    f.write("## Train Augmentation\n\n")
    f.write("- Resize to 256\n")
    f.write("- RandomResizedCrop to 224\n")
    f.write("- Horizontal flip with probability 0.5\n")
    f.write("- Small rotation up to 10 degrees\n")
    f.write("- Mild brightness/contrast/saturation/hue changes\n")
    f.write("- ImageNet normalization\n\n")

    f.write("## Validation/Test Transform\n\n")
    f.write("- Resize to 256\n")
    f.write("- CenterCrop to 224\n")
    f.write("- ImageNet normalization\n\n")

    f.write("## Class Weights\n\n")
    f.write("| Class | Train Count | Weight |\n")
    f.write("|---|---:|---:|\n")

    for _, row in class_weights_df.iterrows():
        f.write(
            f"| {row['class_name']} | {int(row['train_count']):,} | {row['class_weight']:.4f} |\n"
        )

md_path

PosixPath('/backup/Intern/combine_skindiseaseclassifier_devraj/markdown_reports/balancing_and_augmentation_summary.md')

In [16]:
import subprocess

subprocess.run(
    ["python3", str(PROJECT_ROOT / "scripts" / "update_project_structure.py")],
    cwd=PROJECT_ROOT,
    check=True
)

Updated: /backup/Intern/combine_skindiseaseclassifier_devraj/PROJECT_STRUCTURE.txt


CompletedProcess(args=['python3', '/backup/Intern/combine_skindiseaseclassifier_devraj/scripts/update_project_structure.py'], returncode=0)